# LangChain 프롬프트 구성


## OpenAI LLM 준비

- 환경 변수(`.env` 파일)에서 API Key 로딩

## 역할 기반 메시지를 이용한 프롬프트 구성

모든 Role 메시지는 **BaseMessage**가 부모 클래스

| 역할 (Role) | 클래스명        | 설명                                                      |
| ----------- | --------------- | --------------------------------------------------------- |
| system      | `SystemMessage` | 모델의 행동 지침이나 문맥을 설정하는 초기 메시지          |
| user        | `HumanMessage`  | 사용자의 입력을 나타냄 (사람이 입력한 메시지)             |
| assistant   | `AIMessage`     | LLM의 응답을 나타냄                                       |
| tool        | `ToolMessage`   | 도구 실행 결과나 함수 응답을 나타냄 (OpenAI tool 사용 시) |


In [ ]:
# 환경 변수 설정 확인

import os
from dotenv import load_dotenv

## .env 파일 로드
load_dotenv()

## 환경 변수에서 OPENAI_API_KEY 가져오기
open_api_key = os.getenv("OPENAI_API_KEY")
print(f"{open_api_key[:9]}***")

In [ ]:
# OpenAI LLM 준비

from langchain_openai import ChatOpenAI

## LLM 모델 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

print(llm.model_name)

In [ ]:
# 프롬프트 메시지 구성

from langchain_core.messages import SystemMessage, HumanMessage

question = "명탐정 코난에서 쿠도 신이치가 작아진 이유가 뭐죠?"

messages = [
    SystemMessage(
        content="당신은 '명탐정 코난' 전문가입니다. 질문에 친절하고 정확하게 답해 주세요."
    ),
    HumanMessage(content=question),
]

# 프롬프트 메시지 출력해서 구조 확인하기
print(messages)

In [ ]:
# LLM에 메시지 전달하여 응답 받기

response = llm.invoke(messages)

print(f"AI 응답: {response.content}")

## 프롬프트 템플릿을 이용한 메시지 구성

- 프롬프트 변수: 중괄호{} 내부적으로 파이썬 f-string 같은 포맷팅 문법을 사용하기 때문에, "{...}" 안의 내용을 변수 placeholder 로 해석
- system 프롬프트에 설명 목적으로 {} 를 넣으면 LangChain 이 “변수” 영역으로 착각해서 오류 발생 --> {{ }} 형식으로 이스케이프 처리


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 '명탐정 코난' 전문가입니다. 질문에 친절하고 정확하게 답해 주세요.",
        ),
        ("human", "{question}"),  # 변수는 중괄호로 처리
    ]
)

question = "블랙 조직의 리더는 누구인가요?"
prompt_messages = prompt.format_messages(question=question)

response = llm.invoke(prompt_messages)

print(f"AI 응답: {response.content}")

## 역할 메시지로 **변수 바인딩 `(X)`**


In [ ]:
# 리스트 안에 역할 메시지 객체를 담아 템플릿화
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="당신은 '명탐정 코난' 전문가입니다. 질문에 친절하고 정확하게 답해 주세요."
        ),
        HumanMessage(content="{question}"),  # 변수는 중괄호로 처리
    ]
)

question = "블랙 조직의 리더는 누구인가요?"
prompt_messages = prompt.format_messages(question=question)

response = llm.invoke(prompt_messages)

print(f"AI 응답: {response.content}")
print(f"프롬프트 메시지 확인(변수 바인딩 X): {prompt_messages}")

## 역할 메시지 템플릿을 통한 **변수 바인딩`(O)`**


In [ ]:
# 리스트 안에 역할 메시지 템플릿을 통해 프롬프트를 구성해야 함.

from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            "당신은 '명탐정 코난' 전문가입니다. 질문에 친절하고 정확하게 답해 주세요."
        ),
        HumanMessagePromptTemplate.from_template("{question}"),
    ]
)

question = "블랙 조직의 리더는 누구인가요?"
prompt_messages = prompt.format_messages(question=question)

response = llm.invoke(prompt_messages)

print(f"AI 응답: {response.content}")
print(f"프롬프트 메시지 확인(변수 바인딩 O): {prompt_messages}")